In [1]:
import json
import pandas as pd
from pathlib import Path

# Load the JSON file
json_path = Path('sample_data.json')
with open(json_path, 'r') as f:
    data = json.load(f)

# Convert to DataFrame - keeping the key as an index/column
df = pd.DataFrame.from_dict(data, orient='index')

print(f"DataFrame shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst row:")
print(df.iloc[0])


DataFrame shape: (7247, 6)

Columns: ['vis_query', 'chart', 'hardness', 'db_id', 'vis_obj', 'nl_queries']

First row:
vis_query     {'vis_part': 'Visualize SCATTER', 'data_part':...
chart                                                   Scatter
hardness                                                 Medium
db_id                                                activity_1
vis_obj       {'chart': 'scatter', 'x_name': 'FacID', 'y_nam...
nl_queries    [Show the faculty id of each faculty member, a...
Name: 3, dtype: object


In [2]:
df

,vis_query,chart,hardness,db_id,vis_obj,nl_queries
3,"{'vis_part': 'Visualize SCATTER', 'data_part':...",Scatter,Medium,activity_1,"{'chart': 'scatter', 'x_name': 'FacID', 'y_nam...","[Show the faculty id of each faculty member, a..."
4,"{'vis_part': 'Visualize PIE', 'data_part': {'s...",Pie,Medium,activity_1,"{'chart': 'pie', 'x_name': 'Rank', 'y_name': '...",[Show all the faculty ranks and the number of ...
5,"{'vis_part': 'Visualize BAR', 'data_part': {'s...",Bar,Medium,activity_1,"{'chart': 'bar', 'x_name': 'Rank', 'y_name': '...",[Show all the faculty ranks and the number of ...
6,"{'vis_part': 'Visualize PIE', 'data_part': {'s...",Pie,Easy,activity_1,"{'chart': 'pie', 'x_name': 'Building', 'y_name...",[Show all the buildings along with the number ...
7,"{'vis_part': 'Visualize BAR', 'data_part': {'s...",Bar,Easy,activity_1,"{'chart': 'bar', 'x_name': 'Building', 'y_name...",[Show all the buildings along with the number ...
...,...,...,...,...,...,...
3069@y_name@DESC,"{'vis_part': 'Visualize BAR', 'data_part': {'s...",Bar,Hard,twitter_1,"{'chart': 'bar', 'x_name': 'name', 'y_name': '...",[Find the name of each user and number of twee...
2724@x_name@ASC,"{'vis_part': 'Visualize BAR', 'data_part': {'s...",Bar,Medium,school_player,"{'chart': 'bar', 'x_name': 'Denomination', 'y_...",[Create a bar chart showing the total number a...
2724@x_name@DESC,"{'vis_part': 'Visualize BAR', 'data_part': {'s...",Bar,Medium,school_player,"{'chart': 'bar', 'x_name': 'Denomination', 'y_...","[For each denomination, return the denominatio..."
2724@y_name@ASC,"{'vis_part': 'Visualize BAR', 'data_part': {'s...",Bar,Medium,school_player,"{'chart': 'bar', 'x_name': 'Denomination', 'y_...","[For each denomination, return the denominatio..."


In [3]:
# Count of distinct db_id
distinct_db_ids = df['db_id'].nunique()
print(f"Count of distinct db_ids: {distinct_db_ids}")

# Total rows for each db_id
db_id_stats = df.groupby('db_id').size().reset_index(name='total_rows')
print("\nTotal rows per db_id:")
print(db_id_stats)

Count of distinct db_ids: 152

Total rows per db_id:
                 db_id  total_rows
0           activity_1          60
1             aircraft          18
2            allergy_1          84
3    apartment_rentals         124
4         architecture           9
..                 ...         ...
147             wine_1          79
148     workshop_paper          24
149            world_1          27
150           wrestler          19
151              wta_1          18

[152 rows x 2 columns]


In [5]:
# Summary: count of distinct db_ids and rows per db_id
db_id_summary = df.groupby('db_id').size().reset_index(name='total_rows')
db_id_summary = db_id_summary.sort_values('total_rows', ascending=False)
print(f"Distinct db_ids: {len(db_id_summary)}")
print(f"Total rows across all db_ids: {len(df)}")
print("\nBreakdown by db_id (sorted by total_rows descending):")
print(db_id_summary)


Distinct db_ids: 152
Total rows across all db_ids: 7247

Breakdown by db_id (sorted by total_rows descending):
                     db_id  total_rows
70                    hr_1         945
81           manufactory_1         446
143  university_basketball         282
20               college_1         214
21               college_2         139
..                     ...         ...
94               orchestra           6
131              student_1           6
98             perpetrator           6
37     customer_deliveries           5
145                voter_2           4

[152 rows x 2 columns]


In [6]:
db_id_summary

,db_id,total_rows
70,hr_1,945
81,manufactory_1,446
143,university_basketball,282
20,college_1,214
21,college_2,139
...,...,...
94,orchestra,6
131,student_1,6
98,perpetrator,6
37,customer_deliveries,5


In [7]:
import sqlite3

conn = sqlite3.connect('hr_database.db')
cursor = conn.cursor()

# Read and execute the SQL file
with open('hr_schema.sql', 'r') as f:
    sql = f.read()
    
# Remove backticks (MySQL syntax)
sql = sql.replace('`', '')

# Split statements and execute
for statement in sql.split(';'):
    statement = statement.strip()
    if statement:
        try:
            cursor.execute(statement)
        except sqlite3.Error as e:
            print(f"Skipping statement: {e}")

conn.commit()

# Verify
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
print(f"Tables created: {[t[0] for t in tables]}")

conn.close()

Tables created: ['regions', 'countries', 'departments', 'jobs', 'employees', 'job_history', 'locations']


In [8]:
# after creating the database you can open and query it explicitly
import sqlite3
import pandas as pd

conn = sqlite3.connect('hr_database.db')

# list tables using sqlite_master
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("tables:")
print(tables)

# load one table into pandas (e.g. employees)
employees_df = pd.read_sql_query("SELECT * FROM employees LIMIT 5", conn)
print("\nemployees sample:")
print(employees_df)

# you can run arbitrary SQL
dept_counts = pd.read_sql_query(
    "SELECT department_id, COUNT(*) AS cnt FROM employees GROUP BY department_id ORDER BY cnt DESC", conn
)
print("\ndepartment counts:")
print(dept_counts)

conn.close()

tables:
          name
0      regions
1    countries
2  departments
3         jobs
4    employees
5  job_history
6    locations

employees sample:
   EMPLOYEE_ID FIRST_NAME LAST_NAME     EMAIL  PHONE_NUMBER   HIRE_DATE  \
0          100     Steven      King     SKING  515.123.4567  1987-06-17   
1          101      Neena   Kochhar  NKOCHHAR  515.123.4568  1987-06-18   
2          102        Lex   De Haan   LDEHAAN  515.123.4569  1987-06-19   
3          103  Alexander    Hunold   AHUNOLD  590.423.4567  1987-06-20   
4          104      Bruce     Ernst    BERNST  590.423.4568  1987-06-21   

    JOB_ID  SALARY  COMMISSION_PCT  MANAGER_ID  DEPARTMENT_ID  
0  AD_PRES   24000               0           0             90  
1    AD_VP   17000               0         100             90  
2    AD_VP   17000               0         100             90  
3  IT_PROG    9000               0         102             60  
4  IT_PROG    6000               0         103             60  

department counts

In [9]:
hr_rows = df[df['db_id'] == 'hr_1']
print(f"{len(hr_rows)} rows found")
hr_rows.head()

945 rows found


,vis_query,chart,hardness,db_id,vis_obj,nl_queries
1536,"{'vis_part': 'Visualize BAR', 'data_part': {'s...",Bar,Medium,hr_1,"{'chart': 'bar', 'x_name': 'HIRE_DATE', 'y_nam...",[For all employees in the same department and ...
1537,"{'vis_part': 'Visualize BAR', 'data_part': {'s...",Bar,Medium,hr_1,"{'chart': 'bar', 'x_name': 'HIRE_DATE', 'y_nam...",[For all employees in the same department and ...
1538,"{'vis_part': 'Visualize BAR', 'data_part': {'s...",Bar,Extra Hard,hr_1,"{'chart': 'bar', 'x_name': 'HIRE_DATE', 'y_nam...",[For all employees who have the letters D or S...
1539,"{'vis_part': 'Visualize BAR', 'data_part': {'s...",Bar,Extra Hard,hr_1,"{'chart': 'bar', 'x_name': 'HIRE_DATE', 'y_nam...",[For all employees who have the letters D or S...
1540,"{'vis_part': 'Visualize BAR', 'data_part': {'s...",Bar,Extra Hard,hr_1,"{'chart': 'bar', 'x_name': 'HIRE_DATE', 'y_nam...",[For all employees who have the letters D or S...


In [10]:
hr_rows

,vis_query,chart,hardness,db_id,vis_obj,nl_queries
1536,"{'vis_part': 'Visualize BAR', 'data_part': {'s...",Bar,Medium,hr_1,"{'chart': 'bar', 'x_name': 'HIRE_DATE', 'y_nam...",[For all employees in the same department and ...
1537,"{'vis_part': 'Visualize BAR', 'data_part': {'s...",Bar,Medium,hr_1,"{'chart': 'bar', 'x_name': 'HIRE_DATE', 'y_nam...",[For all employees in the same department and ...
1538,"{'vis_part': 'Visualize BAR', 'data_part': {'s...",Bar,Extra Hard,hr_1,"{'chart': 'bar', 'x_name': 'HIRE_DATE', 'y_nam...",[For all employees who have the letters D or S...
1539,"{'vis_part': 'Visualize BAR', 'data_part': {'s...",Bar,Extra Hard,hr_1,"{'chart': 'bar', 'x_name': 'HIRE_DATE', 'y_nam...",[For all employees who have the letters D or S...
1540,"{'vis_part': 'Visualize BAR', 'data_part': {'s...",Bar,Extra Hard,hr_1,"{'chart': 'bar', 'x_name': 'HIRE_DATE', 'y_nam...",[For all employees who have the letters D or S...
...,...,...,...,...,...,...
1962@y_name@DESC,"{'vis_part': 'Visualize BAR', 'data_part': {'s...",Bar,Extra Hard,hr_1,"{'chart': 'bar', 'x_name': 'JOB_ID', 'y_name':...",[Give me a bar chart that groups and count the...
1963@x_name@ASC,"{'vis_part': 'Visualize BAR', 'data_part': {'s...",Bar,Hard,hr_1,"{'chart': 'bar', 'x_name': 'DEPARTMENT_NAME', ...",[Give the name of each department and the numb...
1963@x_name@DESC,"{'vis_part': 'Visualize BAR', 'data_part': {'s...",Bar,Hard,hr_1,"{'chart': 'bar', 'x_name': 'DEPARTMENT_NAME', ...",[Give the name of each department and the numb...
1963@y_name@ASC,"{'vis_part': 'Visualize BAR', 'data_part': {'s...",Bar,Hard,hr_1,"{'chart': 'bar', 'x_name': 'DEPARTMENT_NAME', ...",[Give the name of each department and the numb...
